In [ ]:
import os
import pandas as pd
import sqlalchemy
import numpy as np
from sqlalchemy import text
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 DB 접속 정보 로드

engine = sqlalchemy.create_engine(
    "mysql+pymysql://{user}:{password}@{host}:{port}/{dbname}?charset=utf8".format(
        user     = os.getenv("DB_USER"),
        password = os.getenv("DB_PASSWORD"),
        host     = os.getenv("DB_HOST"),
        port     = os.getenv("DB_PORT"),
        dbname   = os.getenv("DB_NAME"),
    )
)

## STEP 1. 주문 소요시간 분석

로그인 → 주문 완료까지 소요 시간을 개선 전후로 비교합니다.

In [ ]:
# orders: 주문 내역 / login_logs: 로그인 이력
# 개선 전(2025-11~12월) vs 개선 후(2026-01~02월) 기간 설정
query_order = """
SELECT user_id, order_id, order_date
FROM orders
WHERE order_date BETWEEN '2025-11-01' AND '2026-02-28'
  AND status != ''
"""
order = pd.read_sql(query_order, engine)

query_login = """
SELECT user_id, log_date
FROM login_logs
WHERE log_date BETWEEN '2025-11-01' AND '2026-02-28'
"""
login = pd.read_sql(query_login, engine)

In [62]:
order = order.sort_values('order_date')
login = login.sort_values('log_date')

merged = pd.merge_asof(
    order,
    login,
    left_on="order_date",
    right_on="log_date",
    by="user_id",
    direction="backward"
)

merged = merged.dropna(subset=['log_date'])

In [ ]:
merged.head(10)

In [64]:
merged['login_to_order_min'] = (
    merged['order_date'] - merged['log_date']
).dt.total_seconds() / 60

In [ ]:
merged['login_to_order_min'].quantile([0.5,0.75,0.9,0.95])

In [66]:
merged = merged[
    (merged['login_to_order_min'] >= 0) &
    (merged['login_to_order_min'] <= 30)
]

In [ ]:
merged

In [68]:
# 로그인 이후 첫 주문만 사용
merged = merged.sort_values(['user_id','order_date'])

merged['order_rank'] = merged.groupby(['user_id','log_date']).cumcount() + 1

merged = merged[merged['order_rank'] == 1]

In [ ]:
merged

In [70]:
merged['period'] = merged['order_date'].apply(
    lambda x: 'before' if x < pd.Timestamp('2026-01-01') else 'after'
)

In [ ]:
result = merged.groupby('period')['login_to_order_min'].agg([
    'count',
    'mean',
    'median'
])

print(result)

## STEP 2. 기존 고객 vs 신규 고객 비교

In [ ]:
# 가입일 기준으로 기존 고객(2026-01 이전 가입) / 신규 고객 분류
query_user = """
SELECT id AS user_id, wdate AS signup_date
FROM members
"""
user = pd.read_sql(query_user, engine)

In [74]:
merged = merged.merge(user, on='user_id', how='left')

In [75]:
cutoff_date = pd.Timestamp('2026-01-01')

merged['user_type'] = merged['signup_date'].apply(
    lambda x: 'existing' if x < cutoff_date else 'new'
)

In [ ]:
result = merged.groupby(['user_type','period'])['login_to_order_min'].agg([
    'count',
    'mean',
    'median'
]).reset_index()

display(result)

In [ ]:
pivot_mean = result.pivot(index='user_type', columns='period', values='mean')
pivot_median = result.pivot(index='user_type', columns='period', values='median')

display(pivot_mean)
display(pivot_median)

## STEP 3. 주문 전환율 분석

로그인 후 30분 이내 주문 완료 여부로 전환율을 측정합니다.

In [ ]:
query_order = """
SELECT user_id, order_id, order_date AS order_time
FROM orders
WHERE order_date BETWEEN '2025-11-01' AND '2026-02-28'
  AND status != ''
"""
order = pd.read_sql(query_order, engine)

query_login = """
SELECT user_id, log_date AS login_time
FROM login_logs
WHERE log_date BETWEEN '2025-11-01' AND '2026-02-28'
"""
login = pd.read_sql(query_login, engine)

In [83]:
login = login.sort_values('login_time')
order = order.sort_values('order_time')

In [84]:
merged = pd.merge_asof(
    login,
    order,
    left_on="login_time",
    right_on="order_time",
    by="user_id",
    direction="forward",   # 로그인 이후 주문
    tolerance=pd.Timedelta("30min")  # 30분 이내 주문만 인정
)

In [ ]:
merged

In [86]:
merged['is_order'] = merged['order_time'].notnull().astype(int)

In [87]:
cutoff = pd.to_datetime("2026-01-01")

merged['period'] = merged['login_time'].apply(
    lambda x: 'before' if x < cutoff else 'after'
)

In [ ]:
conversion = merged.groupby('period')['is_order'].agg(
    total_login='count',
    converted='sum'
)

conversion['conversion_rate'] = conversion['converted'] / conversion['total_login']

print(conversion)

## STEP 4. 주문당 상품 수(AOV) 분석

In [ ]:
# order_items: 주문 내 상품 목록
# 특수 상품코드(배송비 등) 제외하고 실제 상품만 집계
query_order_item = """
SELECT o.order_id, o.user_id, o.order_date AS order_time,
       oi.product_name
FROM order_items AS oi
INNER JOIN orders AS o ON o.order_id = oi.order_id
WHERE o.order_date >= '2025-11-01' AND o.order_date < '2026-03-01'
  AND oi.product_code NOT IN ('SHIPPING_FEE_1', 'SHIPPING_FEE_2')
  AND o.status != ''
"""
order_items_df = pd.read_sql(query_order_item, engine)

In [112]:
order_items['order_time'] = pd.to_datetime(order_items['order_time'])

order_items = order_items.sort_values(['order_id'])

In [113]:
order_cnt = (
    order_items
    .groupby(['order_id', 'user_id', 'order_time'])
    .size()
    .reset_index(name='product_cnt')
)

In [114]:
cutoff = pd.to_datetime("2026-01-01")

order_cnt['period'] = order_cnt['order_time'].apply(
    lambda x: 'before' if x < cutoff else 'after'
)

In [115]:
result = order_cnt.groupby('period')['product_cnt'].agg([
    'count',
    'mean',
    'median'
]).reset_index()

In [ ]:
total = pd.DataFrame({
    'period': ['total'],
    'count': [order_cnt['product_cnt'].count()],
    'mean': [order_cnt['product_cnt'].mean()],
    'median': [order_cnt['product_cnt'].median()]
})

final = pd.concat([result, total], ignore_index=True)

print(final)